In [87]:
import numpy as np
import glob
import matplotlib.pyplot as plt
from legendmeta import LegendMetadata
from dbetto import Props, TextDB, AttrsDict
from tqdm.notebook import tqdm
import polars as pl
import awkward as ak
from pathlib import Path
import lh5
import lgdo

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
version = "v2.1.5"
base = f"/global/cfs/projectdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/{version}/"
scratch_folder = "/pscratch/sd/b/borrfran/sim-v1.1.0-20260401/"
metadata2_path = "/global/homes/b/borrfran/workspace/l200/legend-metadata"
config = Props.read_from(base+"/config.json", subst_pathvar=True)['setups']['l200']['paths']


meta = LegendMetadata(config['metadata'])
meta2 = LegendMetadata(metadata2_path)

timestamp = meta.dataprod.runinfo.p03.r000.phy.start_key

chmap = meta.channelmap(timestamp)
chmap_ak = ak.Array(chmap.group("system").geds.values())
ges_sorted = chmap_ak.name

DET_TYPES = ("BEGe", "COAX", "ICPC", "PPC")
DET_TYPE_MAP = {"B": "BEGe", "C": "COAX", "V": "ICPC", "P": "PPC"}
DET_TYPE_COLOR = {
    "BEGe": "tab:blue",
    "COAX": "tab:orange",
    "ICPC": "tab:green",
    "PPC": "tab:red",
}

simulated_energies = list(range(200, 1100, 100))

datasets_outdir = "../data//v1/parquet/"
dictionaries_dir = "../data/v1/dictionaries"

rawid_by_det_type = Props.read_from(f'{dictionaries_dir}/rawid_by_det_type.yaml')
eres_dict = Props.read_from(f"{dictionaries_dir}/eres_per_det_tot.yaml")


could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f3ab6467970>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f3ab6467510>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f3ab648c1d0>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f3ab648c6d0>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldataprod/config/evt/p14_r0%%_evt_phy_overwrite.yaml, reason: ParserError('while parsing a block mapping', <yaml._yaml.Mark object at 0x7f3ab64b7b50>, 'did not find expected key', <yaml._yaml.Mark object at 0x7f3ab64b73d0>)
could not scan file /global/cfs/cdirs/m2676/users/borrfran/l200/legend-metadata/jldat

# Axio-electric

In [103]:
interaction = "dark-compton"
pq_version = "3"

In [104]:
for ene in tqdm(simulated_energies):
    job_base="fromfile_dark_compton_{ene}keV_hpge_bulk"
    
    job_string = job_base.format(ene=ene)
    gdml_file = f"{scratch_folder}/generated/pars/geom/l200cfg01-{job_string}-tier_stp-geom.gdml"
    stp_files = [str(p) for p in Path(f"{scratch_folder}/generated/tier/stp/{job_string}/").glob(f"l200cfg01-{job_string}-job_*-tier_stp.lh5")]

    _ = assign_detectors_to_vertices(
        gdml = gdml_path,
        lh5_files = stp_files,
        vtx_group = "vtx",
        save = True
    )

  0%|          | 0/9 [00:00<?, ?it/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/20 [00:00<?, ?file/s]

Processing files:   0%|          | 0/5 [00:00<?, ?file/s]

## test write

In [85]:
lh5_file = "../tmp/l200cfg01-electron_200keV_hpge_bulk-job_0005-tier_stp.lh5"

vtx_group = "vtx"
xloc = np.asarray(lh5.read_as(f"{vtx_group}/xloc", lh5_file, "np"))
yloc = np.asarray(lh5.read_as(f"{vtx_group}/yloc", lh5_file, "np"))
zloc = np.asarray(lh5.read_as(f"{vtx_group}/zloc", lh5_file, "np"))

In [100]:
lh5_file = stp_files[0]
detect_vtx = np.asarray(lh5.read_as(f"{vtx_group}/det", lh5_file, "np"))
detect_vtx

array([b'P00662C', b'C00ANG3', b'V04545A', ..., b'P00574B', b'V07647A',
       b'P00574B'], dtype='|S8')

In [102]:
detect_vtx[0]

b'P00662C'

In [89]:
fill_value = "kkhVVB"
shape = len(xloc)
det_names = np.full(shape, fill_value)

data_encoded = det_names.astype("|S6")

det_lgdo = lgdo.Array(nda=data_encoded)
lh5.write(det_lgdo, "det", str(lh5_file), group=vtx_group, wo_mode="append")

In [62]:
gdml_path = '/global/homes/b/borrfran/scratch/sim-v1.1.0-20260401/generated/pars/geom/l200cfg01-electron_200keV_hpge_bulk-tier_stp-geom.gdml'

In [65]:
stp_file

'/pscratch/sd/b/borrfran/sim-v1.1.0-20260401/generated/tier/stp/electron_200keV_hpge_bulk/l200cfg01-electron_200keV_hpge_bulk-job_0000-tier_stp.lh5'

In [39]:
from bosonic_dm.geometry import assign_detectors_to_vertices, build_detector_map


In [48]:
det_dict = build_detector_map(gdml_path)

In [53]:
x, y, z = det_dict['V02160A']['pos']

In [54]:
point = (10-x, 10-y, 10-z)
det_dict['V02160A']['hpge'].is_inside([point])

array([False])

In [47]:
type(det_dict['V02160A']['hpge'])

pygeomhpges.v02160a.V02160A

In [3]:
from pygeomhpges import make_hpge
import pyg4ometry as pg4

reg = pg4.geant4.Registry()

Welcome to JupyROOT 6.28/00


In [4]:
B00000A = make_hpge(chmap['B00000A'], reg)

In [67]:
inside = assign_detectors_to_vertices(
    gdml = gdml_path,
    lh5_file = stp_file,
    vtx_group = "vtx",
    save = False
)

Checking detectors: 100%|██████████| 101/101 [00:03<00:00, 27.02det/s]


In [69]:
for i, det in enumerate(inside):
    if det == 'none':
        print(i)

165097


B00000A: x = 0.21799999475479126; y = 0.0; z = 0.9666104912757874

In [54]:
for det_name in ges_sorted:
    print(f"{det_name}: x = {vertices_ge[det_name]['xloc'].value}; y = {vertices_ge[det_name]['yloc'].value}; z = {vertices_ge[det_name]['zloc'].value}")

V02160A: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.9167605042457581
V02160B: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.8135709762573242
V05261B: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.6919370293617249
V05266A: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.588747501373291
V05266B: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.48555800318717957
V05268B: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.3823685050010681
V05612A: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.27917900681495667
V07647A: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.17598949372768402
V07647B: x = 0.10899999737739563; y = -0.18879354000091553; z = 0.07280000299215317
B00035C: x = -0.016606640070676804; y = -0.22137799859046936; z = 0.9666104912757874
C000RG1: x = -0.016606640070676804; y = -0.22137799859046936; z = 0.8634210228919983
C000RG2: x = -0.016606640070676804; y = -0.22137799859046936; z = 0.760231494903

In [63]:
chmap[det_name]

{'name': 'P00698B',
 'system': 'geds',
 'location': {'string': 11, 'position': 13},
 'daq': {'crate': 1,
  'card': {'id': 8, 'address': '0x380', 'serialno': None},
  'channel': 4,
  'rawid': 1089604},
 'voltage': {'card': {'id': 5, 'serialno': None},
  'channel': 7,
  'filter': {'id': '7', 'channel': None},
  'flange': {'bundle': 'B', 'sub_bundle': '1', 'channel': '7'}},
 'electronics': {'cc4': {'id': 'B2', 'channel': 6},
  'buffer_card': {'raspberry_pi': 1, 'address': 6, 'channel': 6}},
 'type': 'ppc',
 'production': {'manufacturer': 'Ortec',
  'order': 0,
  'crystal': '698',
  'slice': 'B',
  'enrichment': {'val': 0.874, 'unc': 0.005},
  'passivation': True,
  'reprocessing': False,
  'mass_in_g': 562.0,
  'impcc': {'array': {'value_in_1e9e/cm3': [6.5, 0.5],
    'dist_from_contact_in_mm': [0.0, 31.0]},
   'function_pars': {'z0_in_1e10e/cm3': 0,
    'gradient_in_1e10e/cm4': 0,
    'quadratic_in_1e10e/cm5': 0}},
  'delivered': '2013-11-25'},
 'geometry': {'height_in_mm': 31.0,
  'radiu